# 🧱 Lab: Responsible AI — The Limitations You Must Write Down

**Module 11: Ship It: Capstone, Docs, Responsible AI** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **State** where the triage system is wrong, including that it is a demonstration, not production
2. **Map** known AI failure modes onto this repo's tickets, labels, and logs
3. **Audit** data privacy, dataset licensing, and secret hygiene before anyone clones the repo
4. **Write** a LIMITATIONS.md a reviewer can use to decide whether to trust an output

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| Responsible AI Limitations | A clear, written statement of where your AI system is wrong or unreliable, for example, "a demonstration project, not a production system", so users know what not to trust. | This lab |
| Data Privacy and Licensing | Handling PII with consent and retention limits, while respecting dataset licenses, redistribution rights, and allowed uses across collection, training, and deployment. | This lab |
| AI Failure Modes | Common ways AI apps fail in the real world: out-of-schema labels, data and user drift, rate limits, silent fallbacks, and input drift that degrade quality, latency, or cost without obvious errors. | mini-failure-mode-prediction |
| Secret Hygiene | Secrets hygiene is the habit of keeping passwords, API keys, and tokens out of code and logs, sharing them safely, and rotating them regularly so one mistake doesn’t become a breach. | mini-init-git-repo |

## Prerequisites

- **lab-portfolio-readme**: Reader-order README; Catches is allowed to point here
- **mini-failure-mode-prediction**: Out-of-schema labels, drift, rate limits, silent fallback
- **mini-init-git-repo**: `.gitignore` before first commit; revoke and rotate if a key leaked


## Why write it down

A reviewer who finds the failure modes themselves will not hire you. A reviewer who reads LIMITATIONS.md and still understands the numbers will. The required sentence is not humble branding: **this is a demonstration project, not a production system.**


In [1]:
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

from pathlib import Path
import re


## Part 1: Failure modes for *this* pipeline

Reuse the Module 9 list. Attach each mode to a support-ticket behaviour, not to a generic chatbot.


In [2]:
FAILURE_MODES = [
    {
        "mode": "Out-of-schema labels",
        "here": "Model returns 'Refunds' or 'Other' instead of Billing, Technical, Account, General",
        "mitigation": "Validate at the boundary; drop or retry; never silently coerce",
    },
    {
        "mode": "Silent fallback",
        "here": "Parser fails, code writes General with no log flag",
        "mitigation": "Structured log with parse_error=true; do not swallow the exception",
    },
    {
        "mode": "Input drift",
        "here": "Tickets switch from English billing questions to mixed-language password resets",
        "mitigation": "Gold set is English-only; state that in LIMITATIONS; monitor unknown-token rate",
    },
    {
        "mode": "Rate limits",
        "here": "Hosted backend 429s on a 1,000-row batch",
        "mitigation": "Bounded retry with a cap; CLI exit code non-zero when the cap is hit",
    },
    {
        "mode": "Label drift in production",
        "here": "Billing precision was 84% on 50 gold rows; live mix shifts toward Account",
        "mitigation": "Sample-size caveat in the README; do not claim production accuracy",
    },
]

lines = ["| Failure mode | How it shows up in triage | Mitigation |", "|---|---|---|"]
for row in FAILURE_MODES:
    lines.append(f"| {row['mode']} | {row['here']} | {row['mitigation']} |")
md("### Failure modes mapped to this repo\n\n" + "\n".join(lines))


### Failure modes mapped to this repo

| Failure mode | How it shows up in triage | Mitigation |
|---|---|---|
| Out-of-schema labels | Model returns 'Refunds' or 'Other' instead of Billing, Technical, Account, General | Validate at the boundary; drop or retry; never silently coerce |
| Silent fallback | Parser fails, code writes General with no log flag | Structured log with parse_error=true; do not swallow the exception |
| Input drift | Tickets switch from English billing questions to mixed-language password resets | Gold set is English-only; state that in LIMITATIONS; monitor unknown-token rate |
| Rate limits | Hosted backend 429s on a 1,000-row batch | Bounded retry with a cap; CLI exit code non-zero when the cap is hit |
| Label drift in production | Billing precision was 84% on 50 gold rows; live mix shifts toward Account | Sample-size caveat in the README; do not claim production accuracy |

## Part 2: Vague vs usable limitation sentences

A limitation is usable only if a reader can decide: should I trust this output for my ticket?


In [3]:
PAIRS = [
    (
        "The model may make mistakes.",
        "On the 50-row gold set, Account recall is the weak class; Account tickets are often labelled General.",
    ),
    (
        "Not for production.",
        "This is a demonstration project, not a production system. It has no on-call, no SLA, and no human review queue.",
    ),
    (
        "Be careful with data.",
        "Do not paste tickets that contain names, emails, or account IDs. Logs redact user text; retention is the local run folder only.",
    ),
]

lines = ["| Vague (reject) | Usable (keep) |", "|---|---|"]
for vague, usable in PAIRS:
    lines.append(f"| {vague} | {usable} |")
md("### Limitation quality\n\n" + "\n".join(lines))


### Limitation quality

| Vague (reject) | Usable (keep) |
|---|---|
| The model may make mistakes. | On the 50-row gold set, Account recall is the weak class; Account tickets are often labelled General. |
| Not for production. | This is a demonstration project, not a production system. It has no on-call, no SLA, and no human review queue. |
| Be careful with data. | Do not paste tickets that contain names, emails, or account IDs. Logs redact user text; retention is the local run folder only. |

## Part 3: Privacy, licensing, and secret hygiene checklist

> 🧩 **Data privacy and licensing.** Handle PII with consent and retention limits, and respect dataset licenses.
> Watch which checklist items are release blockers.

Tick these against the triage repo. A no on secrets is a release blocker. A no on licence means you cannot publish the CSV.


In [4]:
CHECKLIST = [
    ("pii", "Tickets in the published CSV have no names, emails, phones, or account IDs", True),
    ("consent", "If using your own messages, you wrote them or have permission to publish", True),
    ("retention", "Run logs are local; README says not to upload raw tickets to third-party tools", True),
    ("licence", "Dataset licence allows this use and does not require you to ship customer PII", True),
    ("gitignore", ".env, .venv, and key files are listed in .gitignore", True),
    ("example_env", ".env.example has placeholder names, not real keys", True),
    ("no_hardcoded_key", "Source files have no sk- live keys and no password literals", True),
]

# Plant a secret in a fake file to show the scan catching it
leaky = "OPENAI_API_KEY=sk-live-example-not-real-but-looks-like-one\n"
clean = "OPENAI_API_KEY=your-key-here\n"
key_pattern = re.compile(r"sk-[A-Za-z0-9]{8,}")

def scan(text: str) -> bool:
    """Return True if the text looks clean of key-shaped secrets."""
    return key_pattern.search(text) is None

secret_scan_ok = (not scan(leaky)) and scan(clean)

lines = ["| Check | Rule | Pass? |", "|---|---|---|"]
for key, rule, ok in CHECKLIST:
    lines.append(f"| {key} | {rule} | {'yes' if ok else 'NO'} |")
lines.append(f"| secret_scan | Fixture leak detected; placeholder .env.example is clean | {'yes' if secret_scan_ok else 'NO'} |")

md("### Responsible AI checklist\n\n" + "\n".join(lines))


### Responsible AI checklist

| Check | Rule | Pass? |
|---|---|---|
| pii | Tickets in the published CSV have no names, emails, phones, or account IDs | yes |
| consent | If using your own messages, you wrote them or have permission to publish | yes |
| retention | Run logs are local; README says not to upload raw tickets to third-party tools | yes |
| licence | Dataset licence allows this use and does not require you to ship customer PII | yes |
| gitignore | .env, .venv, and key files are listed in .gitignore | yes |
| example_env | .env.example has placeholder names, not real keys | yes |
| no_hardcoded_key | Source files have no sk- live keys and no password literals | yes |
| secret_scan | Fixture leak detected; placeholder .env.example is clean | NO |

## Part 4: Write LIMITATIONS.md

Utf-8 on Windows. This file is the Catches section, expanded so the README can stay short.


In [5]:
LIMITATIONS = """# Limitations

This is a demonstration project, not a production system.

## Where it is wrong

- **Labels.** Allowed classes are Billing, Technical, Account, and General. Out-of-schema strings are a failure, not a new class.
- **Account vs General.** On the 50-row gold set, Account is the weak class and is often labelled General.
- **Language.** English tickets only. Mixed-language or password-reset dumps are out of scope.
- **Latency.** Mean latency hides the tail. p95 was 2100 ms in the logged run; batch jobs will feel that.
- **Sample size.** Fifty gold rows are directional. Do not quote these percentages as production accuracy.

## Data rights and privacy

- Publish only tickets with no PII (names, emails, phones, account IDs).
- Do not commit raw customer exports.
- Dataset licence: course sample CSV, for teaching use. Do not redistribute as a commercial corpus.
- Logs keep a run id and label; user text is redacted.

## Secrets

- API keys live in `.env`, which is gitignored.
- `.env.example` has placeholders only.
- If a key was ever committed, revoke it, rotate it, and rewrite history as taught in Module 4.

## What we are not claiming

- No on-call, no SLA, no human-review product.
- No guarantee the hosted backend is available or cheap tomorrow.
"""

out_dir = Path("artifacts")
out_dir.mkdir(parents=True, exist_ok=True)
path = out_dir / "LIMITATIONS.md"
path.write_text(LIMITATIONS, encoding="utf-8")
md(f"Wrote `{path.as_posix()}` ({path.stat().st_size} bytes). The required demonstration sentence is line 3.")


Wrote `artifacts/LIMITATIONS.md` (1305 bytes). The required demonstration sentence is line 3.

## 🎯 Summary

### What You Built

You mapped five AI failure modes onto the triage pipeline, replaced vague disclaimers with usable limitation sentences, ran a privacy-licence-secret checklist including a fixture key scan, and wrote `artifacts/LIMITATIONS.md` that states this is a demonstration project, not a production system.

### Key Takeaways

1. **Stated limitations** — name the wrong class, the language, the tail latency, and the gold-set size
2. **Demonstration, not production** — that sentence is a constraint, not a footer slogan
3. **Failure modes** — out-of-schema, silent fallback, drift, and rate limits must appear as ticket behaviours
4. **Data rights** — PII, consent, retention, and dataset licence are release checks, not essays
5. **Secret hygiene** — `.gitignore` plus a scan; a leaked key is revoke, rotate, rewrite
